# Numerical Solution of a Delay Differential Equation (DDE)

## Problem Statement

We study the nonlinear **delay differential equation (DDE)**:

$$h'(u) = -\ln(2)\,(h(u) + 1)\,h(u - 1), \qquad u \in [0,\, U]$$

with **history condition** (the DDE analogue of an initial condition):

$$h(u) = \phi(u), \qquad u \in [-1,\, 0]$$

### What makes this a DDE?

Unlike an ODE, the rate of change `h'(u)` at time `u` depends not only on the *current* value `h(u)` but also on the value **one unit of time in the past**, `h(u − 1)`. This "memory" effect is the defining feature of a delay differential equation and means the entire history on `[−1, 0]` must be prescribed before integration can begin.

### Equilibria

Setting `h'(u) = 0` gives two constant equilibrium solutions:

- **h\*(u) = 0** — the trivial equilibrium
- **h\*(u) = −1** — a non-trivial equilibrium

The experiments in this notebook explore how trajectories starting from different initial histories (`phi = 1` and `phi = −0.5`) converge toward these equilibria over `u ∈ [0, 20]`.

## Numerical Method

### Why DDEs need special treatment

An ordinary differential equation (ODE) only needs the current state `h(u)` to advance the solution. A DDE additionally needs the *past* state `h(u − 1)`, which is not yet part of the solution when we start integrating. The standard fix is to supply a **history function** `phi` that defines `h` on the interval `[−1, 0]` before integration begins.

### Method of Steps

The idea is to convert the DDE into a sequence of ODEs, one per unit interval:

- On `[0, 1]`: the delayed term `h(u−1)` falls entirely inside `[−1, 0]`, so it is just `phi(u−1)` — a *known* function. The DDE becomes a plain ODE.
- On `[1, 2]`: the delayed term falls inside `[0, 1]`, which was just solved. Again a known function.
- Repeat for `[2, 3]`, `[3, 4]`, …

In practice we do **not** implement these intervals separately. Instead we maintain a single array of all computed values and look up `h(u−1)` by **linear interpolation** at each step.

### Forward Euler discretisation

With step size `dt`, the update rule at grid point `u_k` is:

```
h(u_{k+1}) = h(u_k) + dt * f(u_k)
```

where the right-hand side is:

```
f(u_k) = −ln(2) · (h(u_k) + 1) · h(u_k − 1)
```

The delayed value `h(u_k − 1)` is obtained by linearly interpolating the already-computed portion of the solution array.

### Grid alignment

To avoid floating-point drift when computing `u_k − 1`, the delay is snapped to the nearest integer multiple of `dt` at the start. This ensures the delayed index always lands exactly on a grid point or between two known grid points.

## Code Structure Overview

The notebook contains three functions and a main experiment block.

| Component | Role |
|---|---|
| `solve_dde_forward_euler(phi, U, dt, delay)` | Core solver — builds the grid, runs the Forward Euler loop, returns `(u_all, h)` |
| `phi_const(c)` | Helper — produces a constant history function `phi(u) = c` |
| `make_plot(u, h, title, filename, xlim)` | Utility — plots a solution curve and saves it as a PNG |

**Four experiments are run**, varying the initial history and the step size:

| Figure | History `phi(u)` | Step size `dt` | Purpose |
|---|---|---|---|
| 4.1 | 1.0 | 0.01 | Baseline: large positive initial value |
| 4.2 | −0.5 | 0.01 | Approach from the negative side |
| 4.3 | 1.0 | 0.02 | Coarser step — observe accuracy loss |
| 4.4 | 1.0 | 0.01 | Fine step — reference solution for comparison |

Figures 4.3 and 4.4 together illustrate how step-size reduction improves numerical accuracy.

---
## 1. Imports and Constants

We rely on three standard libraries:

- **NumPy** — array operations and `linspace` / `concatenate` for grid construction
- **math** — scalar `log` for the exact value of ln(2)
- **Matplotlib** — plotting and saving figures

`LN2` is computed once here and reused throughout, rather than calling `math.log(2)` inside the tight integration loop.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

# Precompute ln(2) once to avoid repeated calls inside the loop
LN2 = math.log(2)

---
## 2. Core Solver: `solve_dde_forward_euler`

This function contains the entire numerical procedure in five steps:

1. **Align the delay** to the nearest integer multiple of `dt`, preventing floating-point drift when looking up past values.
2. **Build the history grid** `[−delay, 0]` and sample the history function `phi` on it.
3. **Build the forward grid** `(0, U]` and allocate the solution array (initialised to zero, filled in step 5).
4. **Concatenate** history and forward arrays into a single pair `(u_all, h)` so that all past values are always at hand.
5. **March forward** with Forward Euler: at each step retrieve `h(u_k − 1)` via linear interpolation of already-computed values, then apply the update rule.

The inner helper `h_at(tau, idx_current)` performs the linear interpolation and deliberately restricts itself to the computed prefix of the solution array, enforcing causality.

In [ ]:
def solve_dde_forward_euler(phi, U=20.0, dt=0.01, delay=1.0):
    """
    Solve the DDE:
        h'(u) = -(ln2) * (h(u) + 1) * h(u - 1),  u in [0, U]
    with history (initial condition):
        h(u) = phi(u),  u in [-delay, 0]

    Args:
        phi   : callable defining the history function on [-delay, 0]
        U     : right endpoint of the solution interval
        dt    : time step size
        delay : delay value (default 1.0)

    Returns:
        u_all : full time grid (history segment + forward segment)
        h     : array of solution values at each grid point
    """

    # ----------------------------------------------------------
    # Step 1: Align the delay to the nearest integer multiple of dt.
    # This prevents floating-point drift when looking up delayed values.
    # ----------------------------------------------------------
    m = int(round(delay / dt))   # number of grid points in the history segment
    delay = m * dt               # recompute exact delay after alignment

    # ----------------------------------------------------------
    # Step 2: Build the history grid [-delay, 0] with m+1 points,
    # and evaluate the history function phi at each node.
    # ----------------------------------------------------------
    u_hist = np.linspace(-delay, 0.0, m + 1)               # time nodes for history
    h_hist = np.array([phi(u) for u in u_hist], dtype=float)  # h values from phi

    # ----------------------------------------------------------
    # Step 3: Build the forward grid (0, U] with n_steps points.
    # Starts at dt (not 0, since 0 is already the last history node).
    # ----------------------------------------------------------
    n_steps = int(round(U / dt))          # total number of forward steps
    u_fwd = np.linspace(dt, U, n_steps)   # time nodes for the forward segment

    # ----------------------------------------------------------
    # Step 4: Concatenate history and forward arrays into one pair.
    # u_all: full time axis (m+1+n_steps points total)
    # h    : solution array; forward portion initialised to 0, filled in the loop
    # ----------------------------------------------------------
    u_all = np.concatenate([u_hist, u_fwd])
    h = np.concatenate([h_hist, np.zeros(n_steps)])

    # ----------------------------------------------------------
    # Helper: linearly interpolate h at time tau,
    # using only values already computed up to index idx_current.
    # This enforces causality — no future values are used.
    # ----------------------------------------------------------
    def h_at(tau, idx_current):
        """
        Return h(tau) by linear interpolation over the computed prefix
        u_all[0 : idx_current+1], h[0 : idx_current+1].
        """
        prefix_u = u_all[:idx_current + 1]   # known time nodes
        prefix_h = h[:idx_current + 1]        # corresponding known solution values

        # Find insertion index j such that prefix_u[j-1] <= tau < prefix_u[j]
        j = np.searchsorted(prefix_u, tau)

        # Clamp to left boundary
        if j <= 0:
            return prefix_h[0]
        # Clamp to right boundary
        if j >= len(prefix_u):
            return prefix_h[-1]

        # Linear interpolation between the two bracketing nodes
        u0, u1 = prefix_u[j - 1], prefix_u[j]   # left and right time nodes
        h0, h1 = prefix_h[j - 1], prefix_h[j]   # left and right h values
        # h(tau) ≈ h0 + (h1 - h0) * (tau - u0) / (u1 - u0)
        return h0 + (h1 - h0) * (tau - u0) / (u1 - u0)

    # ----------------------------------------------------------
    # Step 5: Forward Euler main loop.
    # k ranges from m (first forward index) to m+n_steps-1 (last).
    # At each step: h[k+1] = h[k] + dt * f(h[k], h[k - delay])
    # ----------------------------------------------------------
    for k in range(m, m + n_steps):
        u_k = u_all[k]          # current time u_k
        h_k = h[k]              # current solution value h(u_k)

        # Retrieve the delayed solution value h(u_k - delay) via interpolation
        h_delay = h_at(u_k - delay, k)

        # Evaluate the right-hand side: f = -(ln2) * (h(u) + 1) * h(u - delay)
        rhs = -LN2 * (h_k + 1.0) * h_delay

        # Forward Euler update: h(u_{k+1}) = h(u_k) + dt * f
        h[k + 1] = h_k + dt * rhs

    # Return the complete time grid and the solution array
    return u_all, h

---
## 3. Helper Functions

Two small utilities support the solver:

**`phi_const(c)`** is a *factory function*: it takes a constant `c` and returns a function `phi(u) = c`. This is a convenient way to express "the solution was identically equal to `c` for all past time" without writing a new function by hand for each experiment.

**`make_plot(u, h, ...)`** wraps the repetitive Matplotlib boilerplate — creating a figure, plotting the curve, adding axis labels, saving, and closing — into a single call. The `y = 0` reference line is included deliberately so that zero-crossings and convergence to the equilibrium `h* = 0` are immediately visible.

In [ ]:
def phi_const(c):
    """
    Factory function that returns a constant history function phi(u) = c.
    Used to set up constant initial conditions (e.g. phi = 1 or phi = -0.5).
    """
    return lambda u: c   # ignore u, always return the constant c


def make_plot(u, h, title, filename, xlim=(-1, 20)):
    """
    Plot the numerical solution and save it as a PNG file.

    Args:
        u        : time node array
        h        : solution value array
        title    : figure title string
        filename : output file path
        xlim     : x-axis display range, default (-1, 20)
    """
    plt.figure()                   # open a new figure window
    plt.plot(u, h)                 # plot the solution curve h(u)
    plt.axhline(0, linewidth=1)    # draw a y=0 reference line to highlight zero crossings
    plt.xlim(*xlim)                # set x-axis limits (unpack the tuple)
    plt.xlabel("u")                # label the horizontal axis
    plt.ylabel("h(u)")             # label the vertical axis
    plt.title(title)               # set the figure title
    plt.tight_layout()             # auto-adjust margins so labels are not clipped
    plt.savefig(filename, dpi=200) # save at 200 DPI for high-resolution output
    plt.close()                    # close the figure to free memory

---
## 4. Numerical Experiments

We run four experiments and save the results as figures.

**Varying the initial history** (Figures 4.1 and 4.2, both with `dt = 0.01`):

- `phi = 1` starts the trajectory well above the equilibrium `h* = 0` and shows oscillatory decay back toward zero.
- `phi = -0.5` starts between the two equilibria `h* = -1` and `h* = 0`, revealing how the trajectory approaches zero from the negative side.

**Varying the step size** (Figures 4.3 and 4.4, both with `phi = 1`):

- `dt = 0.02` (Figure 4.3) uses a coarser grid — fewer points per unit interval, larger truncation error.
- `dt = 0.01` (Figure 4.4) halves the step size, reducing the per-step error by a factor of roughly 2 (first-order method) and producing a visibly smoother, more accurate solution.

Comparing 4.3 and 4.4 side by side is a standard **step-size convergence check**: if the solutions look similar, both step sizes are in the convergent regime; if they differ noticeably, the coarser grid is under-resolved.

In [ ]:
U = 20.0   # solve on [0, 20]

# ---- Figure 4.1: constant history h(u)=1, step size dt=0.01 ----
# Large positive initial value; observe how the solution decays toward equilibrium
u, h = solve_dde_forward_euler(phi_const(1.0), U=U, dt=0.01)
make_plot(u, h,
          "Numerical solution, history h(u)=1 on [-1,0], dt=0.01",
          "fig_4_1.png", xlim=(-1, U))

# ---- Figure 4.2: constant history h(u)=-0.5, step size dt=0.01 ----
# Initial value in (-1, 0); observe convergence to equilibrium from the negative side
u, h = solve_dde_forward_euler(phi_const(-0.5), U=U, dt=0.01)
make_plot(u, h,
          "Numerical solution, history h(u)=-0.5 on [-1,0], dt=0.01",
          "fig_4_2.png", xlim=(-1, U))

# ---- Figure 4.3: constant history h(u)=1, coarser step size dt=0.02 ----
# Compare with Figure 4.4 to show the effect of doubling the step size
u, h = solve_dde_forward_euler(phi_const(1.0), U=U, dt=0.02)
make_plot(u, h,
          "Step size dt=0.02 (history h(u)=1 on [-1,0])",
          "fig_4_3.png", xlim=(-1, U))

# ---- Figure 4.4: constant history h(u)=1, finer step size dt=0.01 ----
# Pair with Figure 4.3 to illustrate improved accuracy with halved step size
u, h = solve_dde_forward_euler(phi_const(1.0), U=U, dt=0.01)
make_plot(u, h,
          "Step size dt=0.01 (history h(u)=1 on [-1,0])",
          "fig_4_4.png", xlim=(-1, U))

print("Saved figures: fig_4_1.png, fig_4_2.png, fig_4_3.png, fig_4_4.png")